In [59]:
import numpy as np
import pandas as pd
import random

In [60]:
def best_var(X, y):
    best_score, best_col, best_thresh = np.inf, None, None

    for col in range(X.shape[1]):
        s = np.sort(np.unique(X[:, col]))
        thresholds = [(s[i] + s[i+1])/2 for i in range(len(s) - 1)]

        for thresh in thresholds:
            score = split_score(X, y, col, thresh)
            if score < best_score:
                best_score, best_col, best_thresh = score, col, thresh
                
    return (best_col, best_score, best_thresh)
        
def split_score(X, y, feature_column, threshold):
    true_array = y[X[:, feature_column] >= threshold]
    false_array = y[X[:, feature_column] < threshold]

    n1, n2 = len(true_array), len(false_array)
    true_pct, false_pct = n1/(n1+n2), n2/(n1+n2)
    score = true_pct * np.var(true_array) + false_pct * np.var(false_array)

    return score

def sigmoid(log_odds):
    res = 1/ (1+ np.exp(-log_odds))
    return res

In [61]:
from __future__ import annotations
from dataclasses import dataclass

@dataclass
class LeafNode:
    res: float

@dataclass
class DecisionNode:
    col: int
    threshold: float
    leftNode: DecisionNode | LeafNode | None = None
    rightNode: DecisionNode | LeafNode | None = None

In [62]:


def _build(X, prev_pct, y, depth = 2, score_cutoff = 0):   
    if not len(X) or not len(y):
        raise IndexError("Length was 0?")
    
    best_col, best_score, thresh = best_var(X, y)
    curr = DecisionNode(best_col, thresh)

    if depth == 0 or np.var(y) - best_score <= score_cutoff:
        numerator = np.sum(y)
        denominator = np.sum(prev_pct * (1 - prev_pct))
        return LeafNode(numerator/denominator)
    else:

        left_x, left_y, left_pct = X[X[:,best_col] >= thresh], y[X[:,best_col] >= thresh], prev_pct[X[:,best_col] >= thresh]
        right_x, right_y, right_pct = X[X[:,best_col] < thresh], y[X[:,best_col] < thresh],prev_pct[X[:,best_col] < thresh]
        curr.leftNode = _build(left_x, left_pct, left_y, depth-1, score_cutoff)
        curr.rightNode = _build(right_x, right_pct, right_y, depth-1, score_cutoff)
    return curr

def predict(X, tree):
    ret = []
    for sample in X:
        curr = tree
        while curr and type(curr) is DecisionNode:
            c = curr.col
            if sample[c] >= curr.threshold:
                curr = curr.leftNode
            else:
                curr = curr.rightNode
        ret.append(curr.res if curr else None)
    return ret

def create_gradient_boost(X, y, depth, num_trees = 3, lr = 0.1):
    ret = [np.log(len(y[y==1])/len(y[y == 0]))]
    for _ in range(num_trees):
        res = ret[0] + lr * gradient_boost_predict(X,ret)
        pct_res = np.array([sigmoid(r) for r in res])
        new_y = y - pct_res
        ret.append(_build(X, pct_res, new_y, depth = depth))
    return ret

def gradient_boost_predict(X, f, lr = 0.1):
    curr = np.zeros(X.shape[0])
    for tree in f[1:]:
        curr += predict(X, tree)
    return curr


In [63]:
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import export_text
from sklearn.ensemble import RandomForestClassifier
from collections import deque

def _display(curr: DecisionNode | LeafNode | None):
    if type(curr) is DecisionNode:
        print(f"col: {curr.col}, threshold: {curr.threshold:.2f}", end = " | ")
    elif type(curr) is LeafNode:
        print(f"res: {curr.res}", end = " | ")
    else:
        print(f"res : {None}")
        
def _display_graph(tree):
    q, newq = deque([tree]), deque([])
    while q or newq:
        while q:
            curr = q.pop()
            _display(curr)
            if not curr or type(curr) is not DecisionNode: continue
            for item in [curr.leftNode, curr.rightNode]:
                newq.appendleft(item)
        q, newq = newq, deque()
        print("", end ="\n")

In [66]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report

X,y = make_classification(n_samples = 200, n_features = 4, random_state = 0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

gbc = GradientBoostingClassifier(loss="log_loss", learning_rate=0.1, n_estimators=100, max_depth=3, criterion="squared_error", random_state=0)
gbc.fit(X_train, y_train)
node = create_gradient_boost(X_train, y_train, depth=3, num_trees=100)

sk_preds = gbc.predict(X_test)
sk_preds_proba = gbc.predict_proba(X_test)[:, 1]
my_preds = gradient_boost_predict(X_test, node)
my_preds = node[0] + 0.1 * my_preds 
my_preds = np.array([sigmoid(s) for s in my_preds])

# print(mean_squared_error(y_test, sk_preds_proba), r2_score(y_test, sk_preds_proba))
# print(mean_squared_error(y_test, my_preds), r2_score(y_test, my_preds))
print(np.max(np.abs(sk_preds_proba - my_preds)))
print(np.sum(sk_preds ^ (my_preds > 0.5)))

print(classification_report(y_test, sk_preds))
print(classification_report(y_test, my_preds > 0.5))


2.3308990830941696e-05
0
              precision    recall  f1-score   support

           0       0.96      0.92      0.94        26
           1       0.92      0.96      0.94        24

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50

              precision    recall  f1-score   support

           0       0.96      0.92      0.94        26
           1       0.92      0.96      0.94        24

    accuracy                           0.94        50
   macro avg       0.94      0.94      0.94        50
weighted avg       0.94      0.94      0.94        50

